# Problem 10 (100 points)

A well-engineered training pipeline includes data augmentation, learning rate scheduling, regularization, and proper evaluation. In this problem, you will implement custom augmentation techniques (cutout, mixup), build a CNN with batch normalization and dropout, wire up a full training loop with cosine annealing, and add early stopping.

We use the following notation in this problem.
- **Cutout**: randomly zero out a square patch of an image during training.
- **Mixup**: blend two training examples and their labels: $(\tilde{x}, \tilde{y}) = (\lambda x_i + (1-\lambda) x_j,\; \lambda y_i + (1-\lambda) y_j)$ with $\lambda \sim \text{Beta}(\alpha, \alpha)$.
- **Cosine annealing**: $\eta_t = \eta_{\min} + \frac{1}{2}(\eta_{\max} - \eta_{\min})(1 + \cos(\frac{t}{T}\pi))$.
- **Early stopping**: halt training when validation performance has not improved for `patience` epochs.

In [ ]:
# Run code in this cell

"""
DO NOT MAKE ANY CHANGE IN THIS CELL.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader, TensorDataset, random_split
import numpy as np

torch.manual_seed(42)

> WARNING !!!
>
- Beyond importing libraries/modules/classes/functions in the preceding cell, you are **NOT allowed to import anything else for the following purposes**:
    - **As a part of your final solution.**
    - **Temporarily import something to assist you to get a solution.**

## Part 1 (15 points, coding task)

**Do the following tasks.**

Implement two augmentation functions.

1. `cutout_tensor(img, size=8)`: Given a tensor image of shape `(C, H, W)`, randomly select a center position and zero out a `size x size` square patch (clipped at image boundaries). Return the modified tensor.

2. `mixup_batch(images, labels, alpha=0.2)`: Given `images` of shape `(B, C, H, W)` and one-hot `labels` of shape `(B, K)`, sample $\lambda \sim \text{Beta}(\alpha, \alpha)$, create a random permutation of the batch, and compute:
   - `mixed_images = lambda * images + (1 - lambda) * images[perm]`
   - `mixed_labels = lambda * labels + (1 - lambda) * labels[perm]`
   - Return `(mixed_images, mixed_labels)`.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

def cutout_tensor(img, size=8):
    """Apply cutout to a single image tensor. img: (C, H, W) -> (C, H, W)"""
    ...

def mixup_batch(images, labels, alpha=0.2):
    """Apply mixup. images: (B,C,H,W), labels: (B,K) one-hot -> (mixed_images, mixed_labels)"""
    ...

""" END OF THIS PART """

In [ ]:
""" VERIFICATION """
torch.manual_seed(42)
img = torch.ones(3, 32, 32)
img_cutout = cutout_tensor(img, size=8)
assert img_cutout.shape == (3, 32, 32)
num_zeros = (img_cutout == 0).sum().item()
assert num_zeros > 0, "Cutout should zero some pixels"
assert num_zeros <= 3 * 8 * 8

images = torch.randn(8, 3, 32, 32)
labels = torch.zeros(8, 10)
labels[torch.arange(8), torch.randint(0, 10, (8,))] = 1.0
mixed_img, mixed_lbl = mixup_batch(images, labels, alpha=0.2)
assert mixed_img.shape == (8, 3, 32, 32)
assert mixed_lbl.shape == (8, 10)
assert torch.allclose(mixed_lbl.sum(dim=1), torch.ones(8), atol=1e-5)
print("Part 1 passed!")

Now let us build the model that will be trained with these augmentations.

## Part 2 (15 points, coding task)

**Do the following tasks.**

Build a CNN for CIFAR-10 with batch normalization and dropout.

Architecture (input $3 \times 32 \times 32$):
```
Conv(3,32,3,pad=1) -> BN -> ReLU -> Conv(32,32,3,pad=1) -> BN -> ReLU -> MaxPool(2) -> Dropout(0.25)
Conv(32,64,3,pad=1) -> BN -> ReLU -> Conv(64,64,3,pad=1) -> BN -> ReLU -> MaxPool(2) -> Dropout(0.25)
Flatten -> Linear(64*8*8, 512) -> ReLU -> Dropout(0.5) -> Linear(512, 10)
```

Store as class `CIFAR10CNN`.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

class CIFAR10CNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        ...
    
    def forward(self, x):
        """x: (B, 3, 32, 32) -> (B, 10)"""
        ...

""" END OF THIS PART """

In [ ]:
""" VERIFICATION """
model = CIFAR10CNN()
x = torch.randn(4, 3, 32, 32)
model.train()
assert model(x).shape == (4, 10)
model.eval()
assert model(x).shape == (4, 10)
params = sum(p.numel() for p in model.parameters())
print(f"Part 2 passed! Parameters: {params:,}")

With the model and augmentations ready, let us build the full training loop.

## Part 3 (20 points, coding task)

**Do the following tasks.**

Implement `train_model(model, train_loader, val_loader, epochs, lr, device)` with:

- Optimizer: Adam with `weight_decay=1e-4`.
- LR scheduler: `torch.optim.lr_scheduler.CosineAnnealingLR` with `T_max=epochs`.
- Train/eval mode switching.
- `torch.no_grad()` during validation.
- Returns dict with `'train_losses'`, `'train_accs'`, `'val_accs'` (lists of floats, one per epoch).

In [ ]:
### WRITE YOUR SOLUTION HERE ###

def train_model(model, train_loader, val_loader, epochs=10, lr=1e-3, device='cpu'):
    """
    Full training pipeline with cosine annealing.
    Returns: dict with 'train_losses', 'train_accs', 'val_accs'
    """
    ...

""" END OF THIS PART """

In [ ]:
""" VERIFICATION """
torch.manual_seed(42)
X_syn = torch.randn(200, 3, 32, 32)
y_syn = torch.randint(0, 10, (200,))
train_ds = TensorDataset(X_syn[:160], y_syn[:160])
val_ds = TensorDataset(X_syn[160:], y_syn[160:])
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32)

model = CIFAR10CNN()
history = train_model(model, train_loader, val_loader, epochs=5, lr=1e-3)

assert len(history['train_losses']) == 5
assert len(history['train_accs']) == 5
assert len(history['val_accs']) == 5
assert history['train_losses'][-1] < history['train_losses'][0], "Loss should decrease"
print(f"Part 3 passed! Final train loss: {history['train_losses'][-1]:.4f}")

Early stopping prevents overfitting by halting training when validation performance plateaus.

## Part 4 (20 points, coding task)

**Do the following tasks.**

Implement `early_stopping_train(model, train_loader, val_loader, max_epochs, patience, lr, device)` that:

1. Trains until `max_epochs` or until validation accuracy has not improved for `patience` consecutive epochs.
2. Tracks the best validation accuracy and saves the best model state dict (via `copy.deepcopy` or `.state_dict()`).
3. After stopping, restores the model to its best weights.
4. Returns a dict with `'train_losses'`, `'val_accs'`, `'best_epoch'` (int), `'best_val_acc'` (float), `'stopped_epoch'` (int).

In [ ]:
### WRITE YOUR SOLUTION HERE ###

def early_stopping_train(model, train_loader, val_loader, max_epochs=50,
                          patience=5, lr=1e-3, device='cpu'):
    """
    Training with early stopping.
    Returns: dict with 'train_losses', 'val_accs', 'best_epoch', 'best_val_acc', 'stopped_epoch'
    """
    ...

""" END OF THIS PART """

In [ ]:
""" VERIFICATION """
torch.manual_seed(42)
model_es = CIFAR10CNN()
result = early_stopping_train(model_es, train_loader, val_loader, max_epochs=20, patience=3, lr=1e-3)

assert 'best_epoch' in result
assert 'best_val_acc' in result
assert 'stopped_epoch' in result
assert result['stopped_epoch'] <= 20
assert isinstance(result['best_val_acc'], float)
print(f"Part 4 passed! Stopped at epoch {result['stopped_epoch']}, "
      f"best val acc: {result['best_val_acc']:.4f} at epoch {result['best_epoch']}")

Finally, let us reflect on the design choices in our training pipeline.

## Part 5 (10 points, non-coding task)

**Do the following tasks (Reasoning is required).**

1. Cosine annealing decreases the learning rate following $\eta_t = \frac{\eta_0}{2}(1 + \cos(\frac{t}{T}\pi))$. Why is this generally better than a constant learning rate? What happens to $\eta$ near the end of training?

2. Cutout zeros out a random square patch during training. Why does this help generalization? How is cutout conceptually related to dropout?

3. Early stopping monitors validation performance and halts when it plateaus. Is early stopping equivalent to L2 regularization? Explain the implicit regularization effect.

### WRITE YOUR SOLUTION HERE ###



""" END OF THIS PART """

## Part 6 (20 points, coding task)

**Do the following tasks.**

Implement **Label Smoothing Cross-Entropy Loss**.

Instead of hard targets $y = [0, 0, 1, 0, \ldots]$, use smoothed targets:

$$y_i^{\text{smooth}} = \begin{cases} 1 - \epsilon + \frac{\epsilon}{K} & \text{if } i = \text{target} \\ \frac{\epsilon}{K} & \text{otherwise} \end{cases}$$

where $\epsilon$ is the smoothing factor and $K$ is the number of classes.

The loss is $L = -\sum_i y_i^{\text{smooth}} \log p_i$ where $p = \text{softmax}(\text{logits})$.

Implement as `LabelSmoothingCE(num_classes, smoothing)`. When `smoothing=0`, it should match `nn.CrossEntropyLoss`.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

class LabelSmoothingCE(nn.Module):
    def __init__(self, num_classes, smoothing=0.1):
        super().__init__()
        ...
    
    def forward(self, logits, targets):
        """logits: (B, K), targets: (B,) -> scalar loss"""
        ...

""" END OF THIS PART """

In [ ]:
""" VERIFICATION """
logits = torch.randn(8, 10, requires_grad=True)
targets = torch.randint(0, 10, (8,))

criterion_ls = LabelSmoothingCE(num_classes=10, smoothing=0.1)
criterion_ce = nn.CrossEntropyLoss()

loss_ls = criterion_ls(logits, targets)
loss_ce = criterion_ce(logits, targets)

assert loss_ls.dim() == 0, "Loss should be scalar"
assert loss_ls.requires_grad

# smoothing=0 should match standard CE
criterion_ls0 = LabelSmoothingCE(num_classes=10, smoothing=0.0)
loss_ls0 = criterion_ls0(logits, targets)
assert torch.allclose(loss_ls0, loss_ce, atol=1e-5), \
    f"With smoothing=0 should match CE: {loss_ls0.item():.6f} vs {loss_ce.item():.6f}"

print(f"Part 6 passed! LS loss: {loss_ls.item():.4f}, CE loss: {loss_ce.item():.4f}")